In [4]:
import tensorflow as tf
from tensorflow.keras import models, Model, layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input: The 128-dim Learned Alpha Vector (from Pre-training)
VIEW1_PAYLOAD_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")

# Labels to map Filename -> Index in NPY
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")

# Stats Data (CSV) for Ablation
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights (Use your best ones)
WEIGHTS_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128

# --- 1. Architecture (Must match FULL_HMVCL weights) ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Reshape (128,) -> (128, 1)
    x = layers.Reshape((input_shape[0], 1))(inputs)

    # Architecture from your "Script Before" (MaxPool 2)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu')(h)
    return Model(inputs, [h, z])

# --- 2. Load & Align (Robust Version) ---
def load_features():
    print("--- Preparing Data ---")
    if not os.path.exists(VIEW1_PAYLOAD_PATH):
        print(f"Error: {VIEW1_PAYLOAD_PATH} not found.")
        return None, None

    # 1. Load Payload (12736, 128)
    X_view1 = np.load(VIEW1_PAYLOAD_PATH).astype('float32')
    print(f"Loaded Raw View 1 Shape: {X_view1.shape}")

    # 2. Load Labels & CLAMP IT
    df_labels = pd.read_csv(RAW_LABELS_PATH)

    # [THE FIX] Clamp labels to match NPY length
    # This prevents accessing index 15296 if data only has 12736
    if len(df_labels) > len(X_view1):
        print(f"Warning: Labels ({len(df_labels)}) > Data ({len(X_view1)}). Truncating labels to match.")
        df_labels = df_labels.iloc[:len(X_view1)]

    filename_map = {name: i for i, name in enumerate(df_labels['filename'])}

    # 3. Load Stats Data
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    # 4. Align
    valid_payloads = []
    aligned_indices = []

    print(f"Aligning {len(df_merged)} stats rows with {len(X_view1)} payloads...")

    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_map:
            # Safe to access now because map indices are < len(X_view1)
            valid_payloads.append(X_view1[filename_map[fname]])
            aligned_indices.append(idx)

    X_payload = np.array(valid_payloads)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    print(f"Final Aligned Data: {df_final.shape}")
    return df_final, X_payload

# --- 3. Run Ablation Task ---
def run_ablation(df, X_payload, target_label, task_name):
    print(f"\n=== Ablation Study: {task_name} ===")

    # 1. Generate Alpha (CNN) Features (Latent)
    # The input is 128-dim raw, output is 128-dim latent
    print("Extracting Latent Alpha Features...")
    cnn = get_cnn_encoder((X_payload.shape[1],))

    if os.path.exists(WEIGHTS_PATH):
        try:
            cnn.load_weights(WEIGHTS_PATH)
            print("Weights loaded.")
        except Exception as e:
            print(f"Warning: Weight loading failed ({e}). Using random weights (Invalid results).")
    else:
        print(f"Warning: Weights not found at {WEIGHTS_PATH}. Using random weights.")

    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    X_alpha = extractor.predict(X_payload, batch_size=128, verbose=0)

    # 2. Define Feature Groups
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app', 'label', 'Binary', 'Category', 'App']

    feat_beta = [c for c in df.columns if c.startswith('beta_') and not any(x in c for x in exclude)]
    feat_gamma = [c for c in df.columns if c.startswith('gamma_') and not any(x in c for x in exclude)]
    feat_fft = [c for c in df.columns if c.startswith('fft_') and not any(x in c for x in exclude)]

    # 3. Define Experiments
    experiments = {
        "Full Hybrid (Alpha+All Stats)": [X_alpha, df[feat_beta], df[feat_gamma], df[feat_fft]],
        "No Flow Dyn (Remove Beta)":     [X_alpha, df[feat_gamma], df[feat_fft]],
        "No Burst (Remove Gamma)":       [X_alpha, df[feat_beta], df[feat_fft]],
        "No Spectral (Remove FFT)":      [X_alpha, df[feat_beta], df[feat_gamma]],
        "No Alpha (Stats Only)":         [df[feat_beta], df[feat_gamma], df[feat_fft]]
    }

    # 4. Run Loop
    for exp_name, feature_list in experiments.items():
        clean_feats = []
        for f in feature_list:
            if isinstance(f, pd.DataFrame):
                vals = f.select_dtypes(include=[np.number]).values.astype('float32')
                clean_feats.append(StandardScaler().fit_transform(vals))
            else:
                clean_feats.append(f) # Already numpy

        if not clean_feats: continue
        X_final = np.concatenate(clean_feats, axis=1)

        # Train/Test
        X_train, X_test, y_train, y_test = train_test_split(
            X_final, target_label, train_size=LABEL_PERCENTAGE,
            random_state=42, stratify=target_label
        )

        # Use Sample Weights for fair comparison
        weights = compute_sample_weight('balanced', y_train)

        clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.05, n_jobs=-1)
        clf.fit(X_train, y_train, sample_weight=weights)

        y_pred = clf.predict(X_test)
        score = f1_score(y_test, y_pred, average='weighted')
        print(f"   {exp_name}: {score:.4f}")

# --- 4. Main ---
def main():
    df, X_pay = load_features()
    if df is None: return

    print("Reconstructing Labels...")
    app_col = next((c for c in df.columns if c.endswith('application')), None)

    if not app_col:
        print("Error: App column not found.")
        return

    final_apps = []
    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        prefix = "VPN" if "vpn" in fname else "NonVPN"
        final_apps.append(f"{prefix}_{row[app_col]}")

    df['temp_app'] = final_apps

    # Target Apps (Hardest Task)
    target_apps = ['VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout', 'VPN_Facebook', 'VPN_YouTube', 'VPN_Email']
    mask = df['temp_app'].isin(target_apps)

    le = LabelEncoder()
    y = le.fit_transform(df.loc[mask, 'temp_app'])

    run_ablation(df[mask], X_pay[mask], y, "VPN Top Apps (20% Labeled)")

if __name__ == "__main__":
    main()

--- Preparing Data ---
Loaded Raw View 1 Shape: (12555, 128)
Aligning 12555 stats rows with 12555 payloads...
Final Aligned Data: (12374, 275)
Reconstructing Labels...

=== Ablation Study: VPN Top Apps (20% Labeled) ===
Extracting Latent Alpha Features...
Weights loaded.
   Full Hybrid (Alpha+All Stats): 0.9324
   No Flow Dyn (Remove Beta): 0.9186
   No Burst (Remove Gamma): 0.9298
   No Spectral (Remove FFT): 0.9261
   No Alpha (Stats Only): 0.9004
